# Loan Default Prediction – ML Zoomcamp Capstone Project

## Problem Statement

Banks and lending institutions need to assess the risk of a borrower defaulting on a loan **before** approving it.  
Incorrect decisions lead either to lost revenue (rejecting good customers) or to financial losses (accepting high-risk customers).

We treat this as a **binary classification** problem:
- **Target**: `Current_loan_status` → `DEFAULT` vs `NO DEFAULT`
- **Goal**: Predict the probability that a new loan application will default.

A trained model can be used in a real-time credit decision engine:
1. Applicant submits application data (income, employment, loan purpose, amount, etc.).
2. The service returns a default probability + a recommended decision (approve / review / reject).
3. Risk teams can also use the model offline for portfolio stress testing and limit setting.

## Dataset
The dataset contains ~32k historical loan applications with demographic, employment, loan and credit-history features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

DATA_PATH = Path('../data/loan_dataset.csv')
df_raw = pd.read_csv(DATA_PATH)
print('Shape:', df_raw.shape)
df_raw.head()

## 1. Data Cleaning & Preparation

In [ ]:
def clean_loan_amount(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('£', '').replace(',', '').replace('"', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

def clean_income(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(',', '').replace('"', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

df = df_raw.copy()

# Clean monetary columns
df['loan_amnt'] = df['loan_amnt'].apply(clean_loan_amount)
df['customer_income'] = df['customer_income'].apply(clean_income)

# Drop rows without target
df = df.dropna(subset=['Current_loan_status'])

# Binary target
df['default'] = (df['Current_loan_status'] == 'DEFAULT').astype(int)

# Historical default: treat missing as "unknown" → map to 0 (no previous default recorded)
df['historical_default'] = df['historical_default'].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

# Age anomalies (min 18, max 100)
df = df[(df['customer_age'] >= 18) & (df['customer_age'] <= 100)]

# Employment duration: extreme outliers (e.g. 123 years)
df.loc[df['employment_duration'] > 50, 'employment_duration'] = np.nan

# Drop identifier and original status column
df = df.drop(columns=['customer_id', 'Current_loan_status'])

print('After cleaning:', df.shape)
print('Default rate: {:.2%}'.format(df['default'].mean()))
df.head()

In [ ]:
print(df.isnull().sum())
print('\nNumeric describe:')
df.describe().T

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.countplot(data=df, x='default', ax=axes[0,0])
axes[0,0].set_title('Target distribution')

sns.boxplot(data=df, x='default', y='customer_income', ax=axes[0,1])
axes[0,1].set_title('Income vs Default')
axes[0,1].set_yscale('log')

sns.boxplot(data=df, x='default', y='loan_amnt', ax=axes[1,0])
axes[1,0].set_title('Loan amount vs Default')

sns.boxplot(data=df, x='default', y='loan_int_rate', ax=axes[1,1])
axes[1,1].set_title('Interest rate vs Default')

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['home_ownership', 'loan_intent', 'loan_grade']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_cols):
    ct = pd.crosstab(df[col], df['default'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax)
    ax.set_title(f'Default rate by {col}')
    ax.set_ylabel('Proportion')
    ax.legend(['No Default', 'Default'])
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of numeric features with target
num_cols = ['customer_age', 'customer_income', 'employment_duration',
            'loan_amnt', 'loan_int_rate', 'term_years', 'historical_default', 'cred_hist_length']
corr = df[num_cols + ['default']].corr()['default'].drop('default').sort_values()
corr.plot(kind='barh', figsize=(8, 5), title='Correlation with default')
plt.tight_layout()
plt.show()
print(corr)

### Key findings from EDA
- **Interest rate** and **loan grade** are the strongest predictors (higher rate / worse grade → higher default).
- **Historical default** is highly informative when present.
- Income and loan amount show moderate relationships; ratio features may help.
- Class imbalance (~21% default) – we will use stratified splits and evaluate with ROC-AUC / PR-AUC.

## 3. Feature Engineering & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Simple engineered features
df['loan_to_income'] = df['loan_amnt'] / (df['customer_income'] + 1)
df['income_per_year_emp'] = df['customer_income'] / (df['employment_duration'].fillna(1) + 1)

feature_cols = [
    'customer_age', 'customer_income', 'home_ownership', 'employment_duration',
    'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'term_years',
    'historical_default', 'cred_hist_length', 'loan_to_income', 'income_per_year_emp'
]

X = df[feature_cols]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print('Train default rate: {:.2%}'.format(y_train.mean()))

## 4. Preprocessing Pipeline & Models

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, RocCurveDisplay
)
from sklearn.model_selection import cross_val_score

numeric_features = [
    'customer_age', 'customer_income', 'employment_duration',
    'loan_amnt', 'loan_int_rate', 'term_years', 'historical_default',
    'cred_hist_length', 'loan_to_income', 'income_per_year_emp'
]
categorical_features = ['home_ownership', 'loan_intent', 'loan_grade']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=150, max_depth=6, learning_rate=0.1, random_state=42
    )
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    # Cross-validation AUC
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    results[name] = {
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std(),
        'test_auc': auc,
        'test_ap': ap,
        'model': pipe
    }
    print(f'{name:20s}  CV AUC={cv_scores.mean():.4f}±{cv_scores.std():.4f}  Test AUC={auc:.4f}  AP={ap:.4f}')

In [ ]:
# Select best model by test AUC
best_name = max(results, key=lambda k: results[k]['test_auc'])
best_model = results[best_name]['model']
print('Best model:', best_name)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f'ROC – {best_name}')
plt.show()

## 5. Hyper-parameter Tuning (best model family)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Tune GradientBoosting (usually strongest on tabular data)
param_dist = {
    'clf__n_estimators': [100, 150, 200, 250],
    'clf__max_depth': [3, 4, 5, 6],
    'clf__learning_rate': [0.05, 0.1, 0.15],
    'clf__min_samples_leaf': [5, 10, 20],
    'clf__subsample': [0.8, 1.0]
}

gb_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', GradientBoostingClassifier(random_state=42))
])

search = RandomizedSearchCV(
    gb_pipe, param_distributions=param_dist,
    n_iter=20, scoring='roc_auc', cv=3,
    random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print('Best CV AUC:', search.best_score_)

final_model = search.best_estimator_
y_prob_final = final_model.predict_proba(X_test)[:, 1]
print('Final test AUC:', roc_auc_score(y_test, y_prob_final))
print('Final test AP :', average_precision_score(y_test, y_prob_final))

## 6. Persist the final model

In [ ]:
import joblib
from pathlib import Path

MODEL_DIR = Path('../model')
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(final_model, MODEL_DIR / 'loan_default_model.joblib')
print('Model saved to', MODEL_DIR / 'loan_default_model.joblib')

# Also save feature list for the API
import json
meta = {
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'feature_cols': feature_cols,
    'best_model': 'GradientBoosting (tuned)',
    'test_auc': float(roc_auc_score(y_test, y_prob_final))
}
with open(MODEL_DIR / 'model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata saved')

## Summary
- Cleaned currency strings, age outliers, missing values.
- Engineered loan-to-income and income-per-employment-year ratios.
- Compared Logistic Regression, Random Forest, Gradient Boosting and HistGradient Boosting.
- Tuned Gradient Boosting with RandomizedSearchCV.
- Final model achieves strong ROC-AUC on held-out test set.
- Model exported for deployment via FastAPI + Docker.